<a href="https://colab.research.google.com/github/Maneshna/AI_LAB/blob/main/AILAB5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment 5: A Search for a Puzzle Solver*
Objective: Solve the 8-puzzle using A* search.
Problem Statement: The 8-puzzle involves sliding tiles to achieve a goal state. Use A*
to solve it.
Tasks:
Define heuristic functions:
● H1: Number of misplaced tiles.
● H2: Sum of Manhattan distances of all tiles from their goal positions.
● Implement A* with both heuristics.
● Compare the performance of the two heuristics in terms of the number of nodes
explored and solution depth.

In [ ]:
import heapq
from typing import Optional

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)

MOVES = {
    0: [1, 3],    1: [0, 2, 4],    2: [1, 5],
    3: [0, 4, 6], 4: [1, 3, 5, 7], 5: [2, 4, 8],
    6: [3, 7],    7: [4, 6, 8],    8: [5, 7]
}


# ─── Heuristics

def h1_misplaced(state: tuple) -> int:
    """Number of misplaced tiles (excludes blank)."""
    return sum(1 for i in range(9) if state[i] != 0 and state[i] != GOAL[i])


def h2_manhattan(state: tuple) -> int:
    """Sum of Manhattan distances of each tile from its goal position."""
    distance = 0
    for i, tile in enumerate(state):
        if tile != 0:
            goal_idx = GOAL.index(tile)
            distance += abs(i // 3 - goal_idx // 3) + abs(i % 3 - goal_idx % 3)
    return distance


# A* Search

def astar(start: tuple, heuristic) -> tuple[Optional[list], int]:
    """
    A* search for the 8-puzzle.

    Args:
        start     : initial state as a 9-tuple (0 = blank)
        heuristic : callable h(state) -> int

    Returns:
        path      : list of states from start to goal (None if unsolvable)
        explored  : total nodes popped from the priority queue
    """
    h = heuristic(start)
    # heap entry: (f, g, state, parent_state)
    heap = [(h, 0, start, None)]
    visited = {}           # state -> (g, parent)
    explored = 0

    while heap:
        f, g, state, parent = heapq.heappop(heap)

        if state in visited:
            continue
        visited[state] = (g, parent)
        explored += 1

        if state == GOAL:
            return reconstruct(visited, state), explored

        blank = state.index(0)
        for neighbor_idx in MOVES[blank]:
            lst = list(state)
            lst[blank], lst[neighbor_idx] = lst[neighbor_idx], lst[blank]
            neighbor = tuple(lst)
            if neighbor not in visited:
                ng = g + 1
                nf = ng + heuristic(neighbor)
                heapq.heappush(heap, (nf, ng, neighbor, state))

    return None, explored


def reconstruct(visited: dict, goal: tuple) -> list:
    """Trace back from goal to start using the visited map."""
    path = []
    state = goal
    while state is not None:
        path.append(state)
        _, parent = visited[state]
        state = parent
    path.reverse()
    return path


#  Display

def render(state: tuple) -> str:
    """Render a puzzle state as a 3x3 grid string."""
    rows = []
    for i in range(0, 9, 3):
        row = []
        for val in state[i:i+3]:
            row.append("_" if val == 0 else str(val))
        rows.append(" ".join(row))
    return "\n".join(rows)


def print_path(path: list, max_steps: int = 5) -> None:
    """Print first and last few steps of the solution path."""
    n = len(path)
    show = list(range(min(max_steps, n))) + (
        list(range(max(max_steps, n - max_steps), n)) if n > max_steps * 2 else []
    )
    show = sorted(set(show))
    prev = -1
    for i in show:
        if i != prev + 1:
            print(f"  ... ({i - prev - 1} steps skipped) ...")
        print(f"  Step {i}:")
        for line in render(path[i]).splitlines():
            print(f"    {line}")
        prev = i


def compare(start: tuple) -> None:
    """Run A* with H1 and H2, print results and comparison."""
    print("=" * 50)
    print("START STATE:")
    for line in render(start).splitlines():
        print(f"  {line}")
    print(f"\nGOAL STATE:")
    for line in render(GOAL).splitlines():
        print(f"  {line}")
    print("=" * 50)

    results = {}
    for name, h in [("H1 (Misplaced Tiles)", h1_misplaced),
                    ("H2 (Manhattan Distance)", h2_manhattan)]:
        path, explored = astar(start, h)
        results[name] = (path, explored)
        print(f"\n{name}")
        print("-" * 50)
        if path:
            print(f"  Solution depth : {len(path) - 1} moves")
            print(f"  Nodes explored : {explored}")
            print(f"\n  Solution path (first/last steps):")
            print_path(path)
        else:
            print("  No solution found.")

    print("\nCOMPARISON")
    print("-" * 50)
    (p1, e1), (p2, e2) = results.values()
    if p1 and p2:
        d1, d2 = len(p1) - 1, len(p2) - 1
        print(f"  {'Metric':<25} {'H1':>8} {'H2':>8} {'Diff (H2-H1)':>13}")
        print(f"  {'-' * 56}")
        print(f"  {'Solution depth':<25} {d1:>8} {d2:>8} {d2 - d1:>+13}")
        print(f"  {'Nodes explored':<25} {e1:>8} {e2:>8} {e2 - e1:>+13}")
        better = "H2" if e2 < e1 else ("H1" if e1 < e2 else "tie")
        print(f"\n  More efficient heuristic: {better}")
        print(f"  (H2 explores fewer nodes because Manhattan distance")
        print(f"   is a tighter lower bound than misplaced tile count)")
    print("=" * 50)


# ─── Puzzles ──────────────────────────────────────────────────────────────────

EASY   = (1, 2, 3, 4, 0, 5, 7, 8, 6)   # 3 moves
MEDIUM = (1, 2, 3, 0, 4, 6, 7, 5, 8)   # ~8 moves
HARD   = (8, 6, 7, 2, 5, 4, 3, 0, 1)   # ~28 moves

if __name__ == "__main__":
    for label, puzzle in [("EASY", EASY), ("MEDIUM", MEDIUM), ("HARD", HARD)]:
        print(f"\n{'#'*50}")
        print(f"  PUZZLE: {label}")
        print(f"{'#'*50}")
        compare(puzzle)
        print()



##################################################
  PUZZLE: EASY
##################################################
START STATE:
  1 2 3
  4 _ 5
  7 8 6

GOAL STATE:
  1 2 3
  4 5 6
  7 8 _

H1 (Misplaced Tiles)
--------------------------------------------------
  Solution depth : 2 moves
  Nodes explored : 3

  Solution path (first/last steps):
  Step 0:
    1 2 3
    4 _ 5
    7 8 6
  Step 1:
    1 2 3
    4 5 _
    7 8 6
  Step 2:
    1 2 3
    4 5 6
    7 8 _

H2 (Manhattan Distance)
--------------------------------------------------
  Solution depth : 2 moves
  Nodes explored : 3

  Solution path (first/last steps):
  Step 0:
    1 2 3
    4 _ 5
    7 8 6
  Step 1:
    1 2 3
    4 5 _
    7 8 6
  Step 2:
    1 2 3
    4 5 6
    7 8 _

COMPARISON
--------------------------------------------------
  Metric                          H1       H2  Diff (H2-H1)
  --------------------------------------------------------
  Solution depth                   2        2            +0
  Nodes